# 69 · Khối 40 cột XGB — đo mức phụ thuộc, rồi khôi phục cho TEST

`52a` **không** dự đoán trên test: nó điền hằng số prevalence (`OOF[TEST] = _prev[None, :]`).
Vì vậy trên test, 40 trong 257 cột EHR **giống hệt nhau ở cả 78.946 ảnh** — vô thông tin.
Train và val thì có dự đoán thật. Đây là bất đối xứng đã nằm trong mọi số từ `53`.

| | làm gì | chạm test? |
|---|---|---|
| **C5–C6** | đo trên val: thay 40 cột bằng đúng hằng số test đang nhận | **không** |
| **C7** | huấn luyện lại 40 XGB, dự đoán test, ghi **file mới** | không (chỉ suy luận) |

Chạy C6 trước. Nếu chênh lệch dưới sàn nhiễu `0.0007` thì **đóng**, khỏi chạy C7.

In [1]:
import os, json, math, time, tarfile, io as _io, re, datetime, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = os.environ.get('CXRLT_DATA', '')
    assert DATA_DIR, ('khong phai Colab va chua dat CXRLT_DATA -> gan thang duong dan `planB_out`. '
                      'Nhanh du phong "./planB_out" cu bao SAI nguyen nhan.')
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RUN_TAG = 'r2a'     # luot CHOT: gop LAT CHON vao train. DUNG resume qua tag cu.
# giu ten bien `RES57` de khong phai sua hang chuc cho; thu muc thi doi.
SRC64 = os.path.join(OUT_DIR, 'stage4_nb64', RUN_TAG)   # nguon trong so cua 64 — CHI DOC
CKPT  = os.path.join(SRC64, 'ckpt')
RES57 = os.path.join(OUT_DIR, 'stage4_nb69')            # thu muc RIENG cua 69 (ghi o day)
P_OOF = os.path.join(OUT_DIR, 'stage4_nb52a', 'xgb_oof.npy')
# TACH theo notebook: hai tien trinh cung `append` mot file qua Drive FUSE se nuot dong
# cua nhau. 53 va 57 chay song song.
P_TOUCH = os.path.join(OUT_DIR, '_test_touches_nb69.jsonl')
def ensure_res():
    '''Drive FUSE danh roi thu muc RONG — goi TRUOC moi cum ghi (43/38c/38d/41a/41c da dinh).'''
    os.makedirs(RES57, exist_ok=True); return RES57      # KHONG makedirs CKPT: do la cua 64
ensure_res()
assert os.path.isdir(OUT_DIR) and os.listdir(OUT_DIR), f'{OUT_DIR} khong doc duoc -> mount hong'
json.dump({'run_tag': RUN_TAG, 'notebook': '69'}, open(os.path.join(RES57, '_run_info.json'), 'w'))
LOCAL = '/content/cxrlt_518'
READY_X = os.path.join(OUT_DIR, 'cxrlt_518_ready')   # kho do 41c C3-bis tao

FT_BLOCKS = (8, 9, 10, 11)      # mo bang 4 block tren. `11`: I_location ~ 0, nen chon theo chi phi.
LR_FT_DIV = 10.0                # LR cho block mo bang = LR/10 — chuan LP-FT, tranh quen pretrain
BB_RUN = 'a2_E3'; SEEDS = (42, 43)
SEED, SIZE, EPOCHS = 42, 518, 10
BATCH, ACCUM = 32, 8                      # EFF 256 — GIU y het 53/56 de so truc tiep
LR, WD, WARMUP, CLIP = 1e-3, 1e-2, 0.05, 5.0
NW = min(16, (os.cpu_count() or 2))
D_MLD, D_ETOK, EHR_CLIP = 512, 256, 10.0
DO_TEST = False     # 69 KHONG cham test o buoc do — chi doc val

# --- rieng cua 69 ---
P_OOF_NEW = os.path.join(RES57, 'xgb_oof_co_test.npy')   # C7 ghi RA DAY, khong de len ban cu
MAX_CKPT  = 6       # so checkpoint dung de do. None = dung het (20 file, cham ~3x).
                    # Con so ta can la CHENH LECH, va no on dinh qua checkpoint.
BS_EVAL   = 32
# `53`/`64` dat nw=0 cho predict vi loader duoc DUNG LAI moi epoch -> spam teardown.
# O 69, loader val chi tao MOT lan roi bo, nen bat worker an toan va go duoc nut co chai
# giai ma JPEG (2 anh/mau x 39,293 mau). RAM = bs x 2 x 3.22MB x NW x prefetch(4).
NW_EVAL   = min(8, (os.cpu_count() or 2))
_ram_gb   = BS_EVAL * 2 * (3 * 518 * 518 * 4 / 1e9) * NW_EVAL * 4
assert _ram_gb < 20, f'DataLoader se an ~{_ram_gb:.0f} GB -> ha NW_EVAL hoac BS_EVAL'

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

BB_SRC = os.path.join(OUT_DIR, 'stage4_nb38a', BB_RUN, 'backbone_ft.pt')
P_MICRO = os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')
P_VOCAB = os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')
for p, hd in ((BB_SRC, '38a'), (P_MICRO, '29c'), (P_VOCAB, '29c'), (P_OOF, '52a')):
    assert os.path.exists(p), f'khong thay {p} -> chay {hd} truoc'

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')
SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'; TE = ~(TR | VA)
_rs = np.random.default_rng(SEED); _su = np.unique(SUB[TR])
_sel = _rs.choice(_su, size=int(round(0.15*len(_su))), replace=False)
SL = TR & np.isin(SUB, _sel)          # van tinh, CHI de bao cao co bao nhieu anh duoc thu hoi
TF_ = TR.copy()                        # <<< R2a: GOP lat chon vao train
print(f'R2a: thu hoi {int(SL.sum()):,} anh lat chon vao train '
      f'({int((TR & ~SL).sum()):,} -> {int(TF_.sum()):,}, +{SL.sum()/max((TR&~SL).sum(),1):.1%})')
print('     LAT CHON GIO NAM TRONG TRAIN -> khong duoc chon bat cu thu gi o luot nay.')
# R2a: SL GIO NAM TRONG TF_ nen giao cua chung DUNG BANG SL — kiem THUAN chieu do,
# thay vi kiem roi nhau nhu cac luot truoc.
assert int((TF_ & SL).sum()) == int(SL.sum()), 'lat chon CHUA duoc gop het vao train'
assert int(TF_.sum()) == int(TR.sum()), 'TF_ phai bang DUNG TR o luot R2a'
assert not (set(SUB[TR]) & set(SUB[VA])), 'benh nhan train ro ri sang VAL'
assert not (set(SUB[TR]) & set(SUB[TE])), 'benh nhan train ro ri sang TEST'
I_TF, I_SL, I_VA, I_TE = (np.where(m)[0] for m in (TF_, SL, VA, TE))
print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,} | '
      f'test {len(I_TE):,}')
print(f'mo bang block {FT_BLOCKS} (LR/{LR_FT_DIV:.0f}) | E={EPOCHS} | '
      f'BATCH {BATCH}x{ACCUM}={BATCH*ACCUM} | seeds {SEEDS}')

Mounted at /content/drive
thiet bi: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM 102 GB | AMP torch.bfloat16
R2a: thu hoi 39,086 anh lat chon vao train (219,785 -> 258,871, +17.8%)
     LAT CHON GIO NAM TRONG TRAIN -> khong duoc chon bat cu thu gi o luot nay.
377,110 anh | khop 258,871 | chon 39,086 | val 39,293 | test 78,946
mo bang block (8, 9, 10, 11) (LR/10) | E=10 | BATCH 32x8=256 | seeds (42, 43)


## C0 · Config — verbatim `64`, chỉ tách thư mục đọc/ghi

In [2]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()
COLS = [c for c in SPEC['Q_FULL'] if c in E_DF.columns]
_m = np.nan_to_num((E_DF.set_index('study_id')[COLS].reindex(_sid)
                    .apply(pd.to_numeric, errors='coerce').to_numpy('float32')), nan=0.0)
# Cot NHI PHAN thi KHONG z-score: p = 1/219,785 -> gia tri chuan hoa 468 -> softmax bao hoa -> NaN
_b = np.array([np.isin(np.unique(_m[:, j]), (0.0, 1.0)).all() for j in range(_m.shape[1])])
_mu = _m[TF_].mean(0, keepdims=True); _sd = _m[TF_].std(0, keepdims=True) + 1e-6
_mu[:, _b] = 0.0; _sd[:, _b] = 1.0
E_ALL = np.clip((_m - _mu) / _sd, -EHR_CLIP, EHR_CLIP).astype('float32')
_keep = [i for i, c in enumerate(COLS) if not re.match(r'^micro_org_', c)]
assert len(COLS) - len(_keep) == 30, f'bo {len(COLS)-len(_keep)} cot micro_org_*, phai la 30'
COLS = [COLS[i] for i in _keep]; EHR_COL = E_ALL[:, _keep]

_MIC = pd.read_parquet(P_MICRO).set_index('study_id')
_VOC = json.load(open(P_VOCAB)); V_MIC = int(_VOC['V']) + 1; PAD_ID = V_MIC - 1
_mic = _MIC.reindex(_sid)
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max())); assert MIC_L <= 64
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64'); MCNT = np.zeros((len(ORDER), MIC_L), 'float32')
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b

XGF = np.load(P_OOF).astype('float32')
assert XGF.shape == (len(ORDER), C) and np.isfinite(XGF).all()
_l = np.log(np.clip(XGF, 1e-6, 1-1e-6) / (1 - np.clip(XGF, 1e-6, 1-1e-6)))
XGB_F = np.clip((_l - _l[TF_].mean(0, keepdims=True)) / (_l[TF_].std(0, keepdims=True) + 1e-6),
                -EHR_CLIP, EHR_CLIP).astype('float32')
FEAT = np.concatenate([EHR_COL, XGB_F], 1)      # `both` = 217 cot EHR + 40 cot XGB (53: tot nhat)
print(f'FEAT {FEAT.shape} = {EHR_COL.shape[1]} EHR + {C} XGB | tui vi khuan V={V_MIC-1} dai {MIC_L}')

_BASE = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_base.parquet'))
_BASE['index_time'] = pd.to_datetime(_BASE.index_time)
_B = (_BASE[_BASE.study_id.isin(set(ORDER.study_id.unique()))]
      .sort_values(['subject_id','index_time']).reset_index(drop=True))
_B['prev_study'] = _B.groupby('subject_id', sort=False).study_id.shift(1)
_PV = dict(zip(_B.study_id.to_numpy(), _B.prev_study.to_numpy()))
_rw = {}
for _i, _s in enumerate(_sid): _rw.setdefault(_s, []).append(_i)
PREV_ROW = np.full(len(ORDER), -1, 'int64')
for _i, _s in enumerate(_sid):
    _p2 = _PV.get(_s, np.nan)
    if not (isinstance(_p2, float) and np.isnan(_p2)): PREV_ROW[_i] = _rw[int(_p2)][0]
_hp = PREV_ROW >= 0
print(f'anh co phim truoc: {_hp.sum():,} / {len(ORDER):,} ({_hp.mean():.1%})  (44a do 67.2%)')
assert _hp.mean() > 0.5 and (PREV_ROW[_hp] != np.arange(len(ORDER))[_hp]).all()

# =============== rieng cua 69: dung bang dac trung "HANG SO" ma TEST dang nhan ===============
# 52a KHONG du doan tren test; no dien HANG SO prevalence:  OOF[TEST] = _prev[None, :]
# => sau logit + chuan hoa, ca 78,946 hang test deu GIONG HET nhau -> 40 cot do vo thong tin.
_i_te = np.where(TE)[0]
_c40  = XGB_F[_i_te[0]].copy()
assert np.allclose(XGB_F[TE], _c40[None, :], atol=1e-6), (
    'hang test KHONG phai hang so -> gia dinh cua 69 SAI, dung chay tiep')
_ncol = EHR_COL.shape[1]
FEAT_C = FEAT.copy(); FEAT_C[:, _ncol:] = _c40[None, :]      # ban "nhu test": 40 cot XGB = hang so
assert FEAT_C.shape == FEAT.shape
assert np.array_equal(FEAT_C[:, :_ncol], FEAT[:, :_ncol]), '217 cot EHR phai GIU NGUYEN'
assert np.array_equal(FEAT_C[TE], FEAT[TE]), 'hang test von da la hang so -> hai ban phai trung'
assert not np.allclose(FEAT_C[VA], FEAT[VA]), 'hang val phai KHAC nhau, neu khong phep do vo nghia'
print(f'FEAT_C dung xong | 40 cot XGB thay bang hang so cua test '
      f'(bien do {np.abs(_c40).min():.3f}..{np.abs(_c40).max():.3f})')
print(f'  val   : |FEAT - FEAT_C| trung binh = {np.abs(FEAT[VA] - FEAT_C[VA]).mean():.4f}')
print(f'  test  : |FEAT - FEAT_C| trung binh = {np.abs(FEAT[TE] - FEAT_C[TE]).mean():.4f}  (phai = 0)')


FEAT (377110, 257) = 217 EHR + 40 XGB | tui vi khuan V=305 dai 10
anh co phim truoc: 253,283 / 377,110 (67.2%)  (44a do 67.2%)
FEAT_C dung xong | 40 cot XGB thay bang hang so cua test (bien do 0.138..2.312)
  val   : |FEAT - FEAT_C| trung binh = 0.1735
  test  : |FEAT - FEAT_C| trung binh = 0.0000  (phai = 0)


## C1 · EHR + túi + phim trước — verbatim `64`, thêm bảng `FEAT_C` "như test"

In [3]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T
import shutil

os.makedirs(LOCAL, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
# Uu tien KHO da resize san (41c C3-bis tao): copy + bung, vai phut. Neu chua co thi stage tu shard
# goc 1022px — cham hon ~10 lan vi phai giai ma + resize 377k anh.
if _n < len(ORDER) * 0.999 and os.path.isdir(READY_X):
    for f in sorted(x for x in os.listdir(READY_X) if x.endswith('.tar')):
        src = os.path.join(READY_X, f); tmp = os.path.join('/content', f)
        try:
            shutil.copy(src, tmp)                        # copy TRUOC — khong mo tar tren mount
            assert os.path.getsize(tmp) == os.path.getsize(src), 'copy NGAN tu Drive'
            with tarfile.open(tmp) as tar: tar.extractall(LOCAL, filter='data')
        except Exception as e:
            print(f'  {f} HONG ({type(e).__name__}) -> bo qua')
        if os.path.exists(tmp): os.remove(tmp)
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
    print(f'hydrate tu kho: {_n:,} / {len(ORDER):,}')
if _n < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')
    assert os.path.isdir(SHARD_DIR), f'khong thay CA {READY_X} lan {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        nm, blob = pl; dst = os.path.join(LOCAL, os.path.basename(nm))
        if os.path.exists(dst): return
        try:
            im = Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC)
        except Exception:
            return                                       # mot anh hong KHONG duoc giet ca lan chay
        tmp = dst + '.tmp'; im.save(tmp, 'JPEG', quality=95); os.replace(tmp, dst)   # NGUYEN TU
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in done], desc='stage', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()     # doc BULK; lap tren mount FUSE se TREO
            assert len(buf) == os.path.getsize(tp), f'{os.path.basename(tp)}: doc NGAN tren mount'
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_n:,} anh'
print(f'anh local: {_n:,} / {len(ORDER):,}')

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
eval_tf = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

_ck = torch.load(BB_SRC, map_location='cpu', weights_only=False)
_sd0 = {k: v.clone() for k, v in AutoModel.from_pretrained('microsoft/rad-dino').state_dict().items()}
backbone = AutoModel.from_pretrained('microsoft/rad-dino')
backbone.load_state_dict(_ck['bb'], strict=True)
with torch.no_grad():
    _d = max((_sd0[k] - v).abs().max().item() for k, v in backbone.state_dict().items())
del _sd0
assert _d > 1e-6, f'backbone GIONG HET pretrain (lech {_d:.1e}) -> `ft` khong duoc nap'
backbone = backbone.to(dev).eval()
for p in backbone.parameters(): p.requires_grad_(False)
D_CXR = backbone.config.hidden_size
LAYERS = backbone.encoder.layer; NL = len(LAYERS)
GRAD_FROM = min(FT_BLOCKS)
assert max(FT_BLOCKS) < NL and list(FT_BLOCKS) == list(range(GRAD_FROM, NL)), (
    f'FT_BLOCKS {FT_BLOCKS} phai la day LIEN TUC den block cuoi ({NL-1}) — neu khong thi phan '
    'dong cung o TREN chung khong dung chung duoc voi arm ft.')
_cfg = backbone.config
_dp = (getattr(_cfg, 'attention_probs_dropout_prob', 0), getattr(_cfg, 'hidden_dropout_prob', 0),
       getattr(_cfg, 'drop_path_rate', 0))
print(f'BACKBONE `ft` 38a/{BB_RUN} | {NL} block | d={D_CXR} | lech pretrain {_d:.3f}')
print(f'dropout (attn, hidden, drop_path) = {_dp}'
      + ('  -> bang 0, che do train/eval khong anh huong' if max(_dp) == 0 else
         '  -> KHAC 0: ban sao PHAI chay .eval() de khop ban goc'))

def _out(r):
    '''Tensor TRAN, khong phai tuple: `r[0]` se lay MAU DAU cua batch va broadcasting che lap loi.'''
    return r[0] if isinstance(r, tuple) else r

class DS(Dataset):
    '''Tra ve CA HAI anh: hien tai va PHIM TRUOC (+ `has`). `ehr_perm` cho TRAO NHAN QUA.'''
    def __init__(s, idx, tf, ehr_perm=None): s.idx, s.tf, s.perm = np.asarray(idx), tf, ehr_perm
    def __len__(s): return len(s.idx)
    def _img(s, i):
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        return s.tf(Image.merge('RGB', (im, im, im)))
    def __getitem__(s, k):
        i = int(s.idx[k]); pr = int(PREV_ROW[i]); x = s._img(i)
        return dict(image=x, image_prev=(s._img(pr) if pr >= 0 else torch.zeros_like(x)),
                    has=torch.tensor(1.0 if pr >= 0 else 0.0), pos=k,
                    y=torch.from_numpy(YM[i]), row=i,
                    erow=(i if s.perm is None else int(s.perm[k])))

def loader(idx, tf, shuffle, bs, perm=None, nw=NW, drop_last=None):
    return DataLoader(DS(idx, tf, perm), batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=True, drop_last=shuffle if drop_last is None else drop_last,
                      persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)

def run_blocks(h, lo, hi, layers=None):
    L = LAYERS if layers is None else layers
    for i in range(lo, hi): h = _out(L[i - (0 if layers is None else lo)](h))
    return h

# --- cong 1: forward CAT DOI phai bang forward NGUYEN VEN ---
_b0 = next(iter(loader(I_VA[:4], eval_tf, False, 4, nw=0))); _x = _b0['image'].to(dev)
with torch.no_grad():
    _full = backbone(pixel_values=_x).last_hidden_state
    _split = backbone.layernorm(run_blocks(run_blocks(backbone.embeddings(_x), 0, GRAD_FROM),
                                           GRAD_FROM, NL))
_e1 = float((_full - _split).abs().max())
assert _e1 < 1e-3, (f'forward CAT DOI lech {_e1:.2e} -> da cat SAI cho trong kien truc HF. '
                    'DUNG chay tiep, moi so sau day vo nghia.')
N_TOK = _full.shape[1] - 1
assert N_TOK == (SIZE // backbone.config.patch_size) ** 2
print(f'cong 1: forward cat doi khop toi {_e1:.1e} | {N_TOK} patch token')

hydrate tu kho: 377,110 / 377,110
anh local: 377,110 / 377,110


preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

BACKBONE `ft` 38a/a2_E3 | 12 block | d=768 | lech pretrain 0.021
dropout (attn, hidden, drop_path) = (0.0, 0.0, 0.0)  -> bang 0, che do train/eval khong anh huong
cong 1: forward cat doi khop toi 0.0e+00 | 1369 patch token


## C2 · Ảnh + backbone — **verbatim `64`**

In [4]:
class GroupFC(nn.Module):
    def __init__(s, C, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a); return s.n2(h + s.ff(h)), w

class MicroBag(nn.Module):
    '''`const=True`: MOT chi so, khong phai MIC_L ban sao (tui that TB 1.7 vi khuan).'''
    def __init__(s, V, d, const=False):
        super().__init__()
        s.const = const; s.w = nn.Parameter(torch.randn(V, d) * d ** -0.5)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))
    def forward(s, ids, w):
        if s.const:
            ids = torch.zeros(ids.shape[0], 1, dtype=torch.long, device=ids.device)
            w = torch.ones(ids.shape[0], 1, device=ids.device)
        return s.mlp(F.embedding_bag(ids, s.w, per_sample_weights=w, mode='sum'))

class ColTokBag(nn.Module):
    '''d_in token COT + 1 TONG HOP + 1 BAG. Chuoi nay CHUA k1 nhu truong hop rieng.'''
    def __init__(s, d_in, d=256, const=False, V=None):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d); s.const = const
        if const: s.c = nn.Parameter(torch.randn(d_in + 1, d) * 0.02)
        s.mb = MicroBag(V, d, const) if V else None
        if s.mb is not None:
            s.bln = nn.LayerNorm(d); s.cln = nn.LayerNorm(d) if const else nn.Identity()
    def forward(s, e, mids=None, mcnt=None):
        if s.const: base = s.c.unsqueeze(0).expand(e.shape[0], -1, -1)
        else:
            col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)
            base = s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))
        if s.mb is None: return base
        return torch.cat([s.cln(base), s.bln(s.mb(mids, mcnt)).unsqueeze(1)], 1)

class Tier1(nn.Module):
    '''Q = 1369 PATCH, K/V = token EHR. `pr` zero-init; `ln_a` chuan hoa DAU VAO cua phep chieu
       (38d: thieu no thi arm doi chung sup 0.2416 -> 0.0681).'''
    def __init__(s, d, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d); s.pr = nn.Linear(d, d)
        nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
    def forward(s, patch, etok):
        a, _ = s.at(s.n(patch), etok, etok, need_weights=False)
        return patch + s.pr(s.ln_a(a))

class Tier1Img(nn.Module):
    '''Phim truoc o muc PATCH, chay o khong gian D_CXR (TRUOC `pin`).
       KHONG tru thang `patch - prior`: hai phim khong duoc dang ky (tu the, do hit vao). '''
    def __init__(s, d, nhead=8, drop=0.1, diff=True):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.nq = nn.LayerNorm(d); s.nk = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d)
        # LAYERSCALE. `pr` zero-init cho dong nhat thuc o buoc 0 nhung KHONG chan duoc toc do
        # lon len: AdamW cap nhat ~lr/buoc/tham so BAT KE gradient nho the nao, nen mot nhanh xuat
        # phat tu 0 lon TUYEN TINH theo buoc x LR. Do `ln_a` chuan hoa `patch - a` (thuong gan 0,
        # phan lon la nhieu) ve chuan don vi, bien do dau ra do TRON VEN `‖pr‖` quyet dinh. Khi no
        # at `patch` thi ML-Decoder doi che do dot ngot -> sup. Da xay ra 3 lan, luon o ep2-4 tuc
        # ngay sau dinh warmup (0.05 x E = nua epoch 1), va `clip` do 0.0% vi gradient VAN binh
        # thuong — van de o BUOC DI, khong o gradient.
        # Nay `pr` khoi tao CHUAN va mot he so LayerScale `1e-4` giu phan du gan 0 luc dau; ca hai
        # deu co gradient that, va he so phai HOC de lon len thay vi lon lên vi quan tinh AdamW.
        s.pr = nn.Linear(d, d)
        s.gamma = nn.Parameter(torch.full((d,), 1e-4))
        s.diff = diff
    def forward(s, patch, prior, has):
        a, _ = s.at(s.nq(patch), s.nk(prior), s.nk(prior), need_weights=False)
        z = s.gamma * s.pr(s.ln_a(patch - a if s.diff else a))
        return patch + z * has.view(-1, 1, 1)      # khong co phim truoc -> mask nhanh di han

class MLDecoder(nn.Module):
    '''Q = C query lop (KHONG hoc, dung theo bai goc). K/V = patch, co the da bi Tier1 dieu bien.'''
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1, d_ehr_in=0, t1=False):
        super().__init__()
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
        s.pin = nn.Linear(d_in, d); s.img = XAttn(d, nhead, ff, drop)
        s.t1 = Tier1(d, nhead, drop) if t1 else None
        if t1: s.ein = nn.Linear(d_ehr_in, d)
        s.out = GroupFC(C, d)
    def forward(s, patch, etok=None, need_w=False):
        q = s.q.weight.unsqueeze(0).expand(patch.shape[0], -1, -1)
        kv = s.pin(patch)
        if s.t1 is not None: kv = s.t1(kv, s.ein(etok))
        h, w = s.img(q, kv, need_w); return s.out(h), w

class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/37/38*/44*/45/52*/53/56. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()
print('kien truc san sang')

kien truc san sang


## C3 · Kiến trúc — **verbatim `64`**

In [5]:
# Bo `tpb_ft`: 57 da dong huong mo bang (G_ft -0.007..-0.010). Giu `mldec` lam doi chung
# anh-thuan — no dung chung forward backbone nen gan nhu mien phi.
ARMS = {
  'mldec':  dict(ehr=False, img=False, ft=False),
  'tpb':    dict(ehr=True,  img=True,  ft=False),
}
FT_ARMS = [t for t, c in ARMS.items() if c['ft']]

def build_arms(seed=SEED):
    M, E, I, U = nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict()
    for tag, cfg in ARMS.items():
        torch.manual_seed(seed)     # gieo lai TRUOC TUNG arm -> khoi tao giong het nhau
        if cfg['ehr']: E[tag] = ColTokBag(FEAT.shape[1], D_ETOK, False, V=V_MIC)
        M[tag] = MLDecoder(C, D_CXR, D_MLD, d_ehr_in=D_ETOK, t1=cfg['ehr'])
        if cfg['img']: I[tag] = Tier1Img(D_CXR)
        if cfg['ft']:
            # BAN SAO RIENG cua block 8-11 + layernorm. Dung chung thi viec cap nhat chung se
            # doi dac trung cua `mldec` va `tpb` -> pha luon hai doi chung.
            U[tag] = nn.ModuleList([copy.deepcopy(LAYERS[i]) for i in FT_BLOCKS]
                                   + [copy.deepcopy(backbone.layernorm)])
            for p in U[tag].parameters(): p.requires_grad_(True)
    return M.to(dev), E.to(dev), I.to(dev), U.to(dev)

MOD, ETK, IMG, UPF = build_arms()
NP_ = {t: sum(p.numel() for m in ([MOD[t]] + [x[t] for x in (ETK, IMG, UPF) if t in x])
              for p in m.parameters() if p.requires_grad) for t in ARMS}
print(f'{"arm":<12} {"tham so":>12} {"trong do block 8-11":>21}')
for t in ARMS:
    nu = sum(p.numel() for p in UPF[t].parameters()) if t in UPF else 0
    print(f'  {t:<10} {NP_[t]:>12,} {nu:>21,}')

_e = torch.randn(4, FEAT.shape[1], device=dev)
_mi = torch.zeros(4, MIC_L, dtype=torch.long, device=dev); _mc = torch.zeros(4, MIC_L, device=dev)
_p = torch.randn(4, N_TOK, D_CXR, device=dev); _pp = torch.randn(4, N_TOK, D_CXR, device=dev)
_h1 = torch.ones(4, device=dev); _h0 = torch.zeros(4, device=dev)
tok = lambda t: ETK[t](_e, _mi, _mc)

assert not any(p.requires_grad for p in backbone.parameters()), 'backbone GOC phai dong cung'
assert not MOD['mldec'].q.weight.requires_grad, 'query_embed dang hoc'
with torch.no_grad():
    m = MOD['tpb'].eval(); l1, _ = m(_p, tok('tpb'))
    _k = m.t1; m.t1 = None; l2, _ = m(_p, tok('tpb')); m.t1 = _k
    assert (l1 - l2).abs().max() < 1e-5, 'Tier1 KHONG zero-init'
    for t in IMG:
        # Cong TUONG DOI thay vi tuyet doi: LayerScale cho dong nhat thuc XAP XI (~1e-4), khong
        # bang chinh xac. Va ti le bien do moi la thu ta thuc su quan tam — chinh no vuot 1 la sup.
        _rel = float((IMG[t](_p, _pp, _h1) - _p).norm() / (_p.norm() + 1e-9))
        assert _rel < 1e-3, f'{t}: Tier1Img nhieu {_rel:.2e} luc khoi tao, phai < 1e-3'
print('cong 2: Tier1 va Tier1Img deu zero-init OK')

# --- cong 3 (ban sao block 8-11) DA GO: luot nay khong co arm `ft` nao ---
# Neu ai do them lai arm ft ma quen cong nay thi phai bao ngay, khong duoc im lang.
assert not FT_ARMS, (f'co arm ft {FT_ARMS} nhung cong 3 (kiem deepcopy) da bi go khoi 64. '
                     'Lay lai tu 57 truoc khi chay.')
print('cong 3: da go (khong co arm ft o luot nay)')
with torch.no_grad():
    _ti = IMG['tpb']; _ti.pr.weight.normal_(0, 0.02)
    _r = float((_ti(_p, _pp*100.0, _h1) - _p).abs().max() / ((_ti(_p, _pp, _h1) - _p).abs().max()+1e-9))
    assert (_ti(_p, _pp, _h0) - _p).abs().max() < 1e-6, 'has=0 KHONG cat duoc nhanh phim truoc'
    nn.init.zeros_(_ti.pr.weight)
assert _r < 3.0, f'phong dai K/V 100x -> bien do ra tang {_r:.1f}x, `ln_a` khong chan duoc'
print(f'cong 4: `ln_a` chan bien do OK ({_r:.2f}x) | has=0 cat sach nhanh OK')
MOD, ETK, IMG, UPF = build_arms()

arm               tham so   trong do block 8-11
  mldec         2,517,032                     0
  tpb           7,330,856                     0
cong 2: Tier1 va Tier1Img deu zero-init OK
cong 3: da go (khong co arm ft o luot nay)
cong 4: `ln_a` chan bien do OK (1.00x) | has=0 cat sach nhanh OK


## C4 · Arms + cổng — **verbatim `64`**

In [6]:
import glob

FILES = sorted(glob.glob(os.path.join(CKPT, 's*_w_ep*.pt')))
assert FILES, f'khong thay checkpoint nao trong {CKPT} -> 64/r2a da chay chua?'
_all = len(FILES)
if MAX_CKPT: FILES = FILES[-MAX_CKPT:]          # lay epoch SAU cung — khoe nhat
print(f'{_all} checkpoint co san, dung {len(FILES)}:')
for f in FILES: print('   ', os.path.basename(f))

HEADS = []
for f in FILES:
    _w = torch.load(f, map_location='cpu', weights_only=False)
    _m, _e, _i, _u = build_arms(seed=int(_w['seed']))
    _m.load_state_dict(_w['mod']); _e.load_state_dict(_w['etk']); _i.load_state_dict(_w['img'])
    for _mod in (_m, _e, _i):
        _mod.eval()
        for _p in _mod.parameters(): _p.requires_grad_(False)
    HEADS.append((os.path.basename(f)[:-3], _m, _e, _i))
print(f'nap {len(HEADS)} dau doc | arms = {list(ARMS)}')

FEAT_T  = torch.from_numpy(FEAT)
FEAT_CT = torch.from_numpy(FEAT_C)
MIDS_T  = torch.from_numpy(MIDS); MCNT_T = torch.from_numpy(MCNT)


def shared_lo(img):
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        return run_blocks(backbone.embeddings(img), 0, GRAD_FROM)


@torch.no_grad()
def upper_frozen(h):
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        return backbone.layernorm(run_blocks(h, GRAD_FROM, NL))[:, 1:].float()


@torch.no_grad()
def head_forward(mod, etk, img, t, pf, ppf, erow, has, ft):
    '''`ft` = bang dac trung EHR: FEAT_T (that) hoac FEAT_CT (hang so nhu test).'''
    cfg = ARMS[t]
    et = etk[t](ft[erow].to(dev), MIDS_T[erow].to(dev), MCNT_T[erow].to(dev)) if cfg['ehr'] else None
    p = img[t](pf, ppf, has) if cfg['img'] else pf
    return mod[t](p, et)[0]


# 'mldec' khong dung EHR nen chi can chay MOT ban -> dung lam SAN "khong co EHR".
JOBS = [(t, v) for t in ARMS for v in ('that', 'hangso') if not (t == 'mldec' and v == 'hangso')]
FT_OF = {'that': FEAT_T, 'hangso': FEAT_CT}
ACC = {(nm, t, v): np.zeros((len(I_VA), C), 'float32') for nm, _, _, _ in HEADS for t, v in JOBS}
print(f'{len(HEADS)} checkpoint x {len(JOBS)} luot dau doc = {len(HEADS)*len(JOBS)} '
      f'luot / lo | backbone chi chay MOT lan moi lo')

print(f'DataLoader: NW {NW_EVAL} | bs {BS_EVAL} | RAM toi da ~{_ram_gb:.0f} GB')
_t0 = time.time()
with torch.no_grad():
    for b in tqdm(loader(I_VA, eval_tf, False, BS_EVAL, nw=NW_EVAL, drop_last=False), desc='val'):
        hl  = shared_lo(b['image'].to(dev, non_blocking=True))
        hlp = shared_lo(b['image_prev'].to(dev, non_blocking=True))
        hs  = b['has'].to(dev)
        pf, ppf = upper_frozen(hl), upper_frozen(hlp)
        pos, erow = b['pos'].numpy(), b['erow']
        for nm, _m, _e, _i in HEADS:
            for t, v in JOBS:
                ACC[(nm, t, v)][pos] = torch.sigmoid(
                    head_forward(_m, _e, _i, t, pf, ppf, erow, hs, FT_OF[v]).float()).cpu().numpy()

assert not any(np.isnan(a).any() for a in ACC.values()), 'co NaN trong du doan'
_d = np.abs(ACC[(HEADS[0][0], 'tpb', 'that')] - ACC[(HEADS[0][0], 'tpb', 'hangso')]).mean()
assert _d > 1e-6, 'hai ban du doan GIONG HET -> FEAT_C khong he duoc dung, phep do vo nghia'
print(f'\nxong {(time.time()-_t0)/60:.1f} phut | |p_that - p_hangso| TB = {_d:.5f}')


20 checkpoint co san, dung 6:
    s43_w_ep4.pt
    s43_w_ep5.pt
    s43_w_ep6.pt
    s43_w_ep7.pt
    s43_w_ep8.pt
    s43_w_ep9.pt
nap 6 dau doc | arms = ['mldec', 'tpb']
6 checkpoint x 3 luot dau doc = 18 luot / lo | backbone chi chay MOT lan moi lo
DataLoader: NW 8 | bs 32 | RAM toi da ~7 GB


val:   0%|          | 0/1228 [00:00<?, ?it/s]


xong 8.6 phut | |p_that - p_hangso| TB = 0.01119


## C5 · Đo trên val — một lượt backbone, nhiều lượt đầu đọc

Patch token **không** phụ thuộc EHR, nên backbone chỉ chạy một lần mỗi lô; hai bản đặc trưng chỉ khác ở đầu đọc. `mldec` không dùng EHR nên chạy một bản, làm sàn "không có EHR".

In [7]:
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o


tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}
YV = YM[I_VA]
ens = lambda t, v: np.mean([ACC[(nm, t, v)] for nm, _, _, _ in HEADS], 0)

P_ = {(t, v): ens(t, v) for t, v in JOBS}
A_ = {k: ap_cols(YV, p) for k, p in P_.items()}

print(f'VAL — ensemble {len(HEADS)} checkpoint (KHONG phai 20 file cua ban trien khai,'
      f' nen so TUYET DOI thap hon; thu ta doc la CHENH LECH)\n')
print(f'{"":<34}{"macro":>9}{"head":>9}{"medium":>9}{"tail":>9}')
_ten = {('tpb', 'that'): 'tpb · EHR THAT        (= val)',
        ('tpb', 'hangso'): 'tpb · 40 cot HANG SO  (= test)',
        ('mldec', 'that'): 'mldec · KHONG co EHR  (san)'}
for k in [('tpb', 'that'), ('tpb', 'hangso'), ('mldec', 'that')]:
    ap = A_[k]; tr = tiers(ap)
    print(f'{_ten[k]:<34}{np.nanmean(ap):>9.4f}' + ''.join(f'{tr[t]:>9.4f}' for t in tr))

D = {t: tiers(A_[('tpb', 'that')])[t] - tiers(A_[('tpb', 'hangso')])[t] for t in TIER}
DM = float(np.nanmean(A_[('tpb', 'that')]) - np.nanmean(A_[('tpb', 'hangso')]))
print(f'\n{"CHENH LECH (that - hang so)":<34}{DM:>+9.4f}' + ''.join(f'{D[t]:>+9.4f}' for t in D))
G = float(np.nanmean(A_[('tpb', 'that')]) - np.nanmean(A_[('mldec', 'that')]))
print(f'{"de doi chieu: G_ctx (tpb - mldec)":<34}{G:>+9.4f}')
print(f'  => khoi 40 cot XGB chiem {100*DM/G if G > 1e-9 else float("nan"):.0f}% dong gop cua ngu canh')

# --- bootstrap ghep cap THEO BENH NHAN ---
_sub_va = SUB[I_VA]; _u = np.unique(_sub_va)
_idx_of = {s: np.where(_sub_va == s)[0] for s in _u}
_rs_b = np.random.default_rng(69); _bs = []
for _ in tqdm(range(200), desc='bootstrap', leave=False):
    _pick = np.concatenate([_idx_of[s] for s in _rs_b.choice(_u, len(_u), replace=True)])
    _bs.append(float(np.nanmean(ap_cols(YV[_pick], P_[('tpb', 'that')][_pick]))
                     - np.nanmean(ap_cols(YV[_pick], P_[('tpb', 'hangso')][_pick]))))
_lo, _hi = np.percentile(_bs, [2.5, 97.5])
print(f'\nbootstrap 200 lan, ghep cap theo benh nhan: {DM:+.4f}  CI95 [{_lo:+.4f}, {_hi:+.4f}]')

SAN = 0.0007        # san nhieu lap lai that (do tu ban sao 41e)
if _lo > SAN:
    print(f'\n=> CO TIN HIEU. Test dang mat khoang {DM:.4f} macro vi 40 cot bi dat hang so.')
    print('   Chay C7 de khoi phuc, roi chay lai suy luan test.')
elif _hi < SAN:
    print(f'\n=> KHONG dang ke (CI nam duoi san {SAN}). DONG huong nay, khoi chay C7.')
else:
    print(f'\n=> CHUA KET LUAN duoc (CI chua san {SAN}). Tang MAX_CKPT roi do lai.')

json.dump(dict(n_ckpt=len(HEADS), delta_macro=DM, ci=[float(_lo), float(_hi)],
               delta_tier={t: float(v) for t, v in D.items()}, g_ctx=G),
          open(os.path.join(ensure_res(), 'do_xgb_val.json'), 'w'), indent=1)


VAL — ensemble 6 checkpoint (KHONG phai 20 file cua ban trien khai, nen so TUYET DOI thap hon; thu ta doc la CHENH LECH)

                                      macro     head   medium     tail
tpb · EHR THAT        (= val)        0.2631   0.4744   0.1916   0.1176
tpb · 40 cot HANG SO  (= test)       0.2578   0.4698   0.1855   0.1125
mldec · KHONG co EHR  (san)          0.2500   0.4615   0.1800   0.1016

CHENH LECH (that - hang so)         +0.0053  +0.0046  +0.0061  +0.0051
de doi chieu: G_ctx (tpb - mldec)   +0.0132
  => khoi 40 cot XGB chiem 41% dong gop cua ngu canh


bootstrap:   0%|          | 0/200 [00:00<?, ?it/s]


bootstrap 200 lan, ghep cap theo benh nhan: +0.0053  CI95 [+0.0037, +0.0071]

=> CO TIN HIEU. Test dang mat khoang 0.0053 macro vi 40 cot bi dat hang so.
   Chay C7 de khoi phuc, roi chay lai suy luan test.


## C6 · Báo cáo + cổng

In [8]:
# CHI chay khi C6 bao CO TIN HIEU.
import xgboost as xgb

# Lat khop CUA 52a = train TRU lat chon. PHAI dung lat nay, khong dung train R2a:
# val da duoc cham bang mo hinh huan luyen tren dung lat nay. Neu test duoc cham boi mot
# mo hinh KHOE HON (nhieu du lieu hon) thi test lai lech khoi val theo chieu nguoc lai.
TF52 = TR & ~SL
_r52 = TF52.sum() / max(TR.sum(), 1)
print(f'lat khop 52a: {TF52.sum():,} anh = {_r52:.1%} cua train   (52a in ra 219,785 = 84.9%)')
assert not (TF52 & SL).any(), 'TF52 van con lat chon trong do'
assert bool((TF52 <= TR).all()), 'TF52 phai nam gon trong train'
assert 0.80 < _r52 < 0.90, f'ti le {_r52:.1%} lech xa 85% cua 52a -> kiem lai SEED / cach lay _sel'

# --- XG: 217 cot EHR + 304 dem vi khuan (verbatim 52a C1) ---
# C1 chuan hoa theo TF_(R2a) = ca train. O day PHAI chuan hoa lai theo TF52, vi do la
# thong ke ma 52a da dung khi sinh cot XGB cho val.
_raw = np.nan_to_num((E_DF.set_index('study_id')[COLS].reindex(_sid)
                      .apply(pd.to_numeric, errors='coerce').to_numpy('float32')), nan=0.0)
_b52 = np.array([np.isin(np.unique(_raw[:, j]), (0.0, 1.0)).all() for j in range(_raw.shape[1])])
_mu = _raw[TF52].mean(0, keepdims=True); _sd = _raw[TF52].std(0, keepdims=True) + 1e-6
_mu[:, _b52] = 0.0; _sd[:, _b52] = 1.0
X52 = np.clip((_raw - _mu) / _sd, -EHR_CLIP, EHR_CLIP).astype('float32')

XB = np.zeros((len(ORDER), V_MIC - 1), 'float32')
for r in range(len(ORDER)):
    m_ = MCNT[r] > 0
    if m_.any(): np.add.at(XB[r], MIDS[r][m_], MCNT[r][m_])
XG = np.concatenate([X52, XB], 1)
assert XG.shape == (len(ORDER), X52.shape[1] + V_MIC - 1) and np.isfinite(XG).all()
print(f'XG {XG.shape} = {X52.shape[1]} cot EHR + {V_MIC-1} dem vi khuan  (52a in ra 217 + 304)')

XDEV = 'cuda' if torch.cuda.is_available() else 'cpu'
PARAMS = dict(objective='binary:logistic', eval_metric='aucpr', max_depth=6, eta=0.1,
              subsample=0.8, colsample_bytree=0.8, tree_method='hist',
              device=XDEV, nthread=os.cpu_count() or 4)
XGB_ROUNDS = 300                                          # y het 52a/46

OOF_OLD = np.load(P_OOF).astype('float32')
OOF_NEW = np.load(P_OOF_NEW).astype('float32') if os.path.exists(P_OOF_NEW) else OOF_OLD.copy()
P_PRG = os.path.join(ensure_res(), 'xgb_test_done.json')
done = set(json.load(open(P_PRG))) if os.path.exists(P_PRG) else set()

D_TE = xgb.DMatrix(XG[TE]); _t0 = time.time()
for j in range(C):
    if str(j) in done: continue
    npos = int(YM[TF52, j].sum())
    if npos < 5:                                          # 52a: khong hoc noi -> hang so prevalence
        OOF_NEW[TE, j] = float(npos) / max(int(TF52.sum()), 1)
    else:
        bst = xgb.train(PARAMS, xgb.DMatrix(XG[TF52], label=YM[TF52, j]),
                        num_boost_round=XGB_ROUNDS, verbose_eval=False)
        OOF_NEW[TE, j] = bst.predict(D_TE)
    done.add(str(j)); ensure_res(); np.save(P_OOF_NEW, OOF_NEW)
    json.dump(sorted(done), open(P_PRG, 'w'))
    print(f'  [{j+1:>2}/{C}] {LAB40[j]:<28} n+={npos:>6,}  {(time.time()-_t0)/60:6.1f} ph', flush=True)

# --- cong: chi hang TEST duoc doi, moi hang khac phai GIU NGUYEN tung bit ---
assert np.array_equal(OOF_NEW[~TE], OOF_OLD[~TE]), (
    'hang train/val da bi doi -> SAI. Mo hinh da huan luyen voi gia tri CU, khong duoc dung.')
assert not np.allclose(OOF_NEW[TE], OOF_OLD[TE]), 'hang test khong doi -> chua du doan gi'
assert np.isfinite(OOF_NEW).all() and (OOF_NEW >= 0).all() and (OOF_NEW <= 1).all()
print(f'\nghi {P_OOF_NEW}')
print(f'  test: do lech chuan theo nhan {OOF_NEW[TE].std(0).mean():.4f} '
      f'(ban cu = 0.0000 vi la hang so)')
print(f'  val : do lech chuan theo nhan {OOF_NEW[VA].std(0).mean():.4f}  (de doi chieu)')

# KHONG cham nhan test o day. Tinh AP cua XGB tren test la MOT LAN CHAM — de danh cho
# luot doc test cuoi cung, doc mot the ca bang.
print('\nBuoc tiep: trong 64 C0 doi P_OOF sang duong dan tren, chay lai C1 + predict(I_TE).')


lat khop 52a: 219,785 anh = 84.9% cua train   (52a in ra 219,785 = 84.9%)
XG (377110, 522) = 217 cot EHR + 305 dem vi khuan  (52a in ra 217 + 304)
  [ 1/40] Adenopathy                   n+= 2,920     0.0 ph
  [ 2/40] Atelectasis                  n+=55,489     0.0 ph
  [ 3/40] Azygos Lobe                  n+=   176     0.1 ph
  [ 4/40] Calcification of the Aorta   n+= 3,670     0.1 ph
  [ 5/40] Cardiomegaly                 n+=63,420     0.1 ph
  [ 6/40] Clavicle Fracture            n+=   147     0.1 ph
  [ 7/40] Consolidation                n+=13,182     0.1 ph
  [ 8/40] Edema                        n+=31,602     0.2 ph
  [ 9/40] Emphysema                    n+= 3,124     0.2 ph
  [10/40] Enlarged Cardiomediastinum   n+=25,136     0.2 ph
  [11/40] Fibrosis                     n+=   981     0.2 ph
  [12/40] Fissure                      n+= 2,295     0.2 ph
  [13/40] Fracture                     n+= 9,900     0.2 ph
  [14/40] Granuloma                    n+= 2,508     0.3 ph
  [15/40] Her

## C7 · Khôi phục 40 cột cho test — **chỉ chạy nếu C6 báo có tín hiệu**